# YOLO Training

## Import Roboflow Dataset

https://universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e/dataset/11#

In [ ]:
# from google.colab import userdata
# api_roboflow = userdata.get('ROBOFLOW_API')

In [ ]:
# from roboflow import Roboflow
# rf = Roboflow(api_key="qPJBOYszIaWb5Xy9QjDX")
# project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
# version = project.version(11)
# dataset = version.download("yolov11")

## Train YOLO Model

In [ ]:
# from ultralytics import YOLO

# # Load a model
# model = YOLO("yolo11n.pt")  # load a pretrained model (recommended for training)

# # Train the model
# results = model.train(
#     data="/content/License-Plate-Recognition-11/data.yaml",
#     epochs=30,
#     imgsz=640,
#     device=0,
#     patience=50,
# )

# Download license Plate Model

In [ ]:
!pip install ultralytics

In [ ]:
import gdown
import os

def download_file_from_google_drive(file_id: str, output_path: str):
    try:
        gdown.download(id=file_id, output=output_path, quiet=False)
        print(f"Successfully downloaded file to: {output_path}")
    except Exception as e:
        print(f"Error downloading file: {e}")

In [ ]:
# Extract file ID from the provided URL
google_drive_url = "https://drive.google.com/file/d/1Zmf5ynaTFhmln2z7Qvv-tgjkWQYQ9Zdw/view"

# The file ID is the part after /d/ and before /view or /edit
file_id = google_drive_url.split('/d/')[1].split('/view')[0]

output_filename = "yolo_license_plate.pt" # ตั้งชื่อไฟล์ที่คุณต้องการบันทึก
output_directory = "content/models" # โฟลเดอร์ที่จะบันทึก
os.makedirs(output_directory, exist_ok=True)

full_output_path = os.path.join(output_directory, output_filename)

print(f"Attempting to download file ID: {file_id} to {full_output_path}")
download_file_from_google_drive(file_id, full_output_path)

Attempting to download file ID: 1Zmf5ynaTFhmln2z7Qvv-tgjkWQYQ9Zdw to content/models/yolo_license_plate.pt


Downloading...
From: https://drive.google.com/uc?id=1Zmf5ynaTFhmln2z7Qvv-tgjkWQYQ9Zdw
To: /content/content/models/yolo_license_plate.pt
100%|██████████| 6.24M/6.24M [00:00<00:00, 239MB/s]

Successfully downloaded file to: content/models/yolo_license_plate.pt


## Preprocess Image Function

In [ ]:
import cv2

def histogram_equalization(image):
  # Apply Histogram Equalization
  equalized_image = cv2.equalizeHist(image)

  return equalized_image


def intensity_transformation(image, alpha=1.5, beta=10):
  new_image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
  return new_image



def contour_image(image):
  edged = cv2.Canny(image, 50, 150)
  contours, _ = cv2.findContours(edged, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
  image = cv2.drawContours(image, contours, -1, (0, 255, 0), 2)
  return image


def filter(image):
  sharpened_image = cv2.filter2D(image, -1, kernel)
  return sharpened_image

## Inferences Model and Cropped Image

In [ ]:
def inference_yolo_and_save_bboxes(model_path: str,
                                   input_folder: str,
                                   output_folder: str,
                                   confidence_threshold: float = 0.5):

  from PIL import Image
  import os
  import numpy as np

  # Get YOLO model
  model = YOLO(model_path)
  os.makedirs(output_folder, exist_ok=True)

  image_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

  for img_name in image_files:
      img_path = os.path.join(input_folder, img_name)
      img = Image.open(img_path).convert("RGB")
      results = model(img, verbose=False) # Run inference

      # Process results for each image
      for r in results:
          for i, box in enumerate(r.boxes.xyxy):
              conf = r.boxes.conf[i]
              if conf >= confidence_threshold:
                  x1, y1, x2, y2 = map(int, box)

                  # Ensure coordinates are within image bounds
                  x1 = max(0, x1)
                  y1 = max(0, y1)
                  x2 = min(img.width, x2)
                  y2 = min(img.height, y2)

                  if x2 > x1 and y2 > y1: # Valid bounding box
                      cropped_img = img.crop((x1, y1, x2, y2))

                      # Preprocess Image
                      cropped_img = np.array(cropped_img)
                      cropped_img = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2GRAY)
                      cropped_img = intensity_transformation(cropped_img)
                      # cropped_img = contour_image(cropped_img)
                      # cropped_img = histogram_equalization(cropped_img)
                      cropped_img = Image.fromarray(cropped_img, 'L')

                      # Save to folder
                      output_filename = f"{os.path.splitext(img_name)[0]}_bbox_{i}_{int(conf*100)}.png"
                      output_path = os.path.join(output_folder, output_filename)
                      cropped_img.save(output_path)
                      print(f"Saved: {output_path}")


In [ ]:
from ultralytics import YOLO

In [ ]:
# Configuration
import os
YOLO_MODEL_PATH = '/content/content/models/yolo_license_plate.pt'
INPUT_IMAGES_FOLDER = '/content/input_images'
OUTPUT_BOUNDING_BOXES_FOLDER = '/content/cropped_image'
CONF_THRESHOLD = 0.5


inference_yolo_and_save_bboxes(YOLO_MODEL_PATH,
                               INPUT_IMAGES_FOLDER,
                               OUTPUT_BOUNDING_BOXES_FOLDER,
                               CONF_THRESHOLD)

Saved: /content/cropped_image/istock-13172924388_bbox_0_80.png
Saved: /content/cropped_image/Screen-Shot-2563-04-17-at-15.03.26_bbox_0_78.png


# EasyOCR

https://github.com/JaidedAI/EasyOCR

In [ ]:
!pip install easyocr

In [ ]:
import easyocr
import pandas as pd
from PIL import Image

# this needs to run only once to load the model into memory
reader = easyocr.Reader(['th', 'en'])

def process_images_to_dataframe(input_folder: str, lang_list: list = ['th']):

    reader = easyocr.Reader(lang_list)

    data = []
    supported_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')

    for img_name in os.listdir(input_folder):
        if img_name.lower().endswith(supported_extensions):
            img_path = os.path.join(input_folder, img_name)

            try:
                with Image.open(img_path) as img:
                    img.verify() # Verify that it is a valid Pillow image

                result = reader.readtext(img_path)

                # Concatenate all detected text for the current image
                extracted_text = " ".join([text for (bbox, text, prob) in result])

                data.append({'image_name': img_name, 'text': extracted_text})
            except Exception as e:
                print(f"Skipping {img_name} due to an error: {e}")
                # Append with empty text if processing fails
                data.append({'image_name': img_name, 'text': ''})

    return pd.DataFrame(data)

In [ ]:
df = process_images_to_dataframe(input_folder=OUTPUT_BOUNDING_BOXES_FOLDER)
df

,image_name,text
0,istock-13172924388_bbox_0_80.png,ทก๓๔รอ กรุงเทพมหานคร
1,Screen-Shot-2563-04-17-at-15.03.26_bbox_0_78.png,กท 2456


# Multimodal Approach

In [ ]:
!pip install transformers accelerate
!pip install -U bitsandbytes
!pip install qwen-vl-utils[decord]==0.0.8

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 122.9 MB/s eta 0:00:00


In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", torch_dtype="auto", device_map="auto"
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")


messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "/content/cropped_image/istock-13172924388_bbox_0_80.png",
            },
            {"type": "text", "text": "Extract the text from image."},
        ],
    }
]


# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['The text in the image is:\n\nไทย ตระกูล\nกรุงเทพมหานคร']
